<a href="https://colab.research.google.com/github/rburchf1/AI102Challenges/blob/main/assignment1_sentiment_ryan_burchfield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — Sentiment Analysis: Classical ML vs. DL vs. LLM

**Ryan Burchfield**

## 1. Overview & dataset selection

Twitter US Airline Sentiment

I selected the Twitter dataset because the sentiment analysis contains 3 options: positive, negative, and neutral. Determining neutrality seems more challenging, in terms of semantic analysis, than a binary setup. In addition, the tertiary framework is more comparable to the type of analysis I might conduct at work.

## 2. Load & inspect data
The Twitter data is in a .csv file on my google drive.

The examples and the class distribution analysis are also provided below.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
!pip install pandas numpy scikit-learn matplotlib seaborn
!pip install torch
!pip install tensorflow keras
!pip install transformers accelerate -U
!pip uninstall -y datasets  # Uninstall datasets to clear any corrupted state
!pip install --upgrade datasets # Reinstall/upgrade datasets
!pip install nltk
!pip install keras
!pip install gensim

Found existing installation: datasets 5.0.1
Uninstalling datasets-5.0.1:
  Successfully uninstalled datasets-5.0.1
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 38.5 MB/s eta 0:00:00


In [9]:
#Install/Import basics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


print('OK: libraries imported')

OK: libraries imported


In [10]:
# Load Twitter Dataset
file_path = '/content/drive/MyDrive/Tweets.csv'
try:
  ds = pd.read_csv(file_path)
  print('File loaded successfully.')
except FileNotFoundError:
  print(f'File not found at path: {file_path}. Please check the file path.')
except Exception as e:
  print(f'An error occurred while loading the file: {e}')

#Generate basic information about dataset
print("DataSet Info:")
ds.info()

#Create dataset description
print("\nDataSet Description:")
display(ds.describe(include='all'))

#View a few rows from the dataset
print("\nFirst 5 rows of the DataSet:")
display(ds.head())

File loaded successfully.
DataSet Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created       

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
count,1.464000e+04,14640,14640.000000,9178,10522.000000,14640,40,14640,32,14640.000000,14640,1019,14640,9907,9820
unique,NaN,3,NaN,10,NaN,6,3,7701,13,NaN,14427,832,14247,3081,85
top,NaN,negative,NaN,Customer Service Issue,NaN,United,negative,JetBlueNews,Customer Service Issue,NaN,@united thanks,"[0.0, 0.0]",2015-02-24 09:54:34 -0800,"Boston, MA",Eastern Time (US & Canada)
freq,NaN,9178,NaN,2910,NaN,3822,32,63,12,NaN,6,164,5,157,3744
mean,5.692184e+17,NaN,0.900169,NaN,0.638298,NaN,NaN,NaN,NaN,0.082650,NaN,NaN,NaN,NaN,NaN
std,7.791112e+14,NaN,0.162830,NaN,0.330440,NaN,NaN,NaN,NaN,0.745778,NaN,NaN,NaN,NaN,NaN
min,5.675883e+17,NaN,0.335000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
25%,5.685592e+17,NaN,0.692300,NaN,0.360600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
50%,5.694779e+17,NaN,1.000000,NaN,0.670600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
75%,5.698905e+17,NaN,1.000000,NaN,1.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN



First 5 rows of the DataSet:


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [11]:
# View class distribution and evaluate the dataset for balance

x = ds['text']
y = ds['airline_sentiment']

print('Sentiment class distribution:')
display(y.value_counts())

Sentiment class distribution:


,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


COMMENTS ON DATASET BALANCE

**airline_sentiment    count**
negative	           9178
neutral	             3099
positive	           2363


As one might expect, the dataset is imbalanced towared the negative. It seems that customers are most likely to spend effort to tweet an experience when it is negative and accept a neutral or good experience as a fair trade requiring no comment since the customer paid for a service. The obvious exception would be an exceptionally good customer service experience.

## 3. Preprocessing
The following pre-processing steps were conducted:

1. Check for emojis throughout the dataset.

2. Decide whether to keep or remove emojis.

3. Split data

4. Make lowercase, remove URLs/usernames (if any), strip spaces, tokenize, remove stopwords, remove puncutation, stemming, and lemmatization.

In [12]:
#EMOJIS
#Detect if there are emojies in the dataset using Regex
#Note: This section of code was generated using the Gemini integration in Google Colab

import re

emoji_pattern = re.compile(
    "["  # Start character set
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Miscellaneous Symbols and Pictographs
    "\U0001F680-\U0001F6FF"  # Transport and Map Symbols
    "\U0001F1E0-\U0001F1FF"  # Regional indicator symbols
    "\U00002600-\U000026FF"  # Miscellaneous Symbols
    "\U00002700-\U000027BF"  # Dingbats
    "]+"
)

def contains_emoji(text):
    return bool(emoji_pattern.search(str(text)))

# Originally sampled some texts to check for emojis but no emojis were found.
# Given the details of the assignment I expected emojis, so I needed to check the whole dataset.
# The sample check was commented out.
#print("Checking for emojis in sample texts:")
#found_emoji = False
#for i, text in enumerate(x.sample(n=10, random_state=42)):
    #if contains_emoji(text):
        #print(f"  Text {i+1} (contains emoji): {text}")
        #found_emoji = True
    #else:
        #print(f"  Text {i+1} (no emoji): {text}")

#if not found_emoji:
    #print("  No emojis found in the selected sample of texts.")
#else:
    #print("  Emojis were found in the sample texts.")

#print("\nNow checking the full dataset (this might take a moment if the dataset is large):")

# Check the entire 'text' column for emojis
num_texts_with_emojis = x.apply(contains_emoji).sum()

if num_texts_with_emojis > 0:
    print(f"The 'text' column contains emojis. Found {num_texts_with_emojis} texts with emojis.")
    print("It would be beneficial to decide whether to remove or preserve them during preprocessing based on the task.")
else:
    print("The 'text' column does not appear to contain emojis.")

The 'text' column contains emojis. Found 487 texts with emojis.
It would be beneficial to decide whether to remove or preserve them during preprocessing based on the task.


***EMOJIS ANALYSIS AND DECISION ON WHETHER TO KEEP OR REMOVE EMOJIS WHEN CLEANING DATA***

The dataset contains 487 entries with emojis, which represent only ~3% of the tweets. Therefore, emojis will be removed.

In [13]:
# Split data for training and validation and testing
from sklearn.model_selection import train_test_split

#FIRST SPLIT
#80% training + validation, 20% for test

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#SECOND SPLIT
#Take 12.5% of the 80% for validation
#12.5% of 80% = 10% of the original dataset

x_train, x_val, y_train, y_val = train_test_split(
    x, y,
    test_size=0.125,
    random_state=42,
    stratify=y
)

In [31]:
#Define and apply a clean function

def basic_clean(text):
    text = str(text).lower()  # Convert to string and lowercase

    # Remove URLs (http/https links)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove mentions (@usernames)
    text = re.sub(r'@\w+', '', text)

    # Remove emojis using the previously defined pattern
    text = emoji_pattern.sub(r'', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function
clean_x_train = x_train.apply(basic_clean)
clean_x_val = x_val.apply(basic_clean)
clean_x_test = x_test.apply(basic_clean)

print("Original text sample:", x_train.iloc[0])
print("Cleaned text sample:", clean_x_train.iloc[0])

print("\nFirst 5 cleaned training texts:")
display(clean_x_train.head())

Original text sample: @VirginAmerica Can't bring up my reservation online using Flight Booking Problems code
Cleaned text sample: can't bring up my reservation online using flight booking problems code

First 5 cleaned training texts:


,text
86,can't bring up my reservation online using fli...
14047,educate bohol is a 501(c)(3) w/all volunteer s...
3642,i mean is there a real live person somewhere i...
2356,how about plowing the snow at a gate before th...
5455,i met my twitter friend waiting outside the tr...


In [15]:
#Additional preprocessing: tokenization, remove stopwords, remove puncutation, stemming, and lemmatization

#Reference: This section of code was adapted from AI102 (NLP Techniques) Module 3 Guided Lab.

import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
import string

nltk.download('stopwords')
nltk.download('wordnet')

text = "Natural Language Processing enables computers to understand human language."

ext = text.translate(str.maketrans('', '', string.punctuation))

# Tokenize (split) the text into separate words.
# text.split() breaks the sentence into a list of words using spaces.
tokens = text.split()

# Get the set of English stop words.
# A set is used because it allows fast checking of "is this word in the list?"
stop_words = set(stopwords.words('english'))

# Remove stop words from our list of tokens.
# This list comprehension means:
# "for each word in tokens, keep it only if it is NOT in stop_words".
filtered_tokens = [word for word in tokens if word not in stop_words]

# Create (initialize) the stemmer and lemmatizer objects that we will use.
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Apply stemming and then lemmatization to each filtered word.
# For each word in filtered_tokens:
#   1. stemmer.stem(word) gets the stem (basic form)
#   2. lemmatizer.lemmatize(...) changes that stem to a dictionary form if possible
# The final results are stored in a new list called processed_tokens.
processed_tokens = [lemmatizer.lemmatize(stemmer.stem(word)) for word in filtered_tokens]

# Print the cleaned version of the text (lowercased and without punctuation)
print("Original Text:")
print(text)

# Print a blank line and then the tokens after removing stop words
print("\nFiltered Tokens (Stop Words Removed):")
print(filtered_tokens)

# Print another blank line and then the tokens after stemming and lemmatization
print("\nProcessed Tokens (Stemmed and Lemmatized):")
print(processed_tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Original Text:
Natural Language Processing enables computers to understand human language.

Filtered Tokens (Stop Words Removed):
['Natural', 'Language', 'Processing', 'enables', 'computers', 'understand', 'human', 'language.']

Processed Tokens (Stemmed and Lemmatized):
['natur', 'languag', 'process', 'enabl', 'comput', 'understand', 'human', 'language.']


## 4. Model A: Classical ML (TF–IDF + Logistic Regression)

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2))
xtr = tfidf.fit_transform(x_train)
xva = tfidf.transform(x_val)
clf = LogisticRegression(max_iter=200)
clf.fit(xtr, y_train)
preds = clf.predict(xva)

print(classification_report(y_val, preds, digits=4))


              precision    recall  f1-score   support

    negative     0.8166    0.9590    0.8821      1147
     neutral     0.7182    0.5387    0.6156       388
    positive     0.8333    0.5424    0.6571       295

    accuracy                         0.8027      1830
   macro avg     0.7894    0.6800    0.7183      1830
weighted avg     0.7985    0.8027    0.7893      1830



In [20]:
#Misclassification in Classic ML (TF-IDF + LR)

# Assuming preds, y_val, x_val, clean_x_val are available from previous cells.
# preds: predictions from the Logistic Regression model on x_val
# y_val: true labels for the validation set
# x_val: original text for the validation set
# clean_x_val: cleaned text for the validation set

# Identify misclassified examples
misclassified_indices_lr = np.where(y_val.values != preds)[0]

print(f"Found {len(misclassified_indices_lr)} misclassified examples in the validation set for TF-IDF + Logistic Regression.\n")

print("--- Sample of Misclassified Examples (TF-IDF + Logistic Regression) ---")
num_samples_to_show = 5

if len(misclassified_indices_lr) == 0:
    print("No misclassified examples found. The model achieved 100% accuracy on the validation set!")
else:
    for i, idx in enumerate(misclassified_indices_lr[:num_samples_to_show]):
        original_text = x_val.iloc[idx]
        cleaned_text = clean_x_val.iloc[idx]
        true_label = y_val.iloc[idx]
        predicted_label = preds[idx]

        print(f"\n--- Misclassification #{i+1} ---")
        print(f"Original Tweet: {original_text}")
        print(f"Cleaned Tweet: {cleaned_text}")
        print(f"True Label: {true_label}")
        print(f"Predicted Label: {predicted_label}")

print("\n--- Analysis of Misclassification Patterns ---")
print("Based on the sample of misclassified examples, here are some potential patterns:")
print("1.  **Sarcasm/Irony:** Tweets that express a negative sentiment using positive-sounding words (or vice-versa) can be tricky. For example, 'I love waiting for 3 hours for my flight' might be classified as positive due to 'love' if the model doesn't capture the sarcastic context.")
print("2.  **Subtle Negativity/Neutrality:** Sometimes, tweets have a subtle negative undertone or are truly neutral but contain words that lean towards another sentiment. Phrases like 'not good' or 'could be better' might be ambiguous.")
print("3.  **Ambiguity:** Some tweets are genuinely ambiguous, even to human annotators. For instance, a question about a flight delay might not express explicit negativity but could be interpreted that way given the common context of airline complaints.")
print("4.  **Keyword Over-reliance:** TF-IDF models heavily rely on the importance of individual words or short phrases. If a 'positive' word appears in a negative context, or a 'negative' word in a neutral context, the model might be misled if it doesn't understand the broader sentence structure.")
print("5.  **Contextual Shift:** The meaning of words can shift based on context. 'Delay' is generally negative in airline context, but if paired with a hopeful or resolved statement, the sentiment can change.")
print("These patterns highlight the challenges of capturing nuanced human language with simpler Bag-of-Words (TF-IDF) models, which often lack the deeper semantic understanding of more complex deep learning or LLM approaches.")

Found 361 misclassified examples in the validation set for TF-IDF + Logistic Regression.

--- Sample of Misclassified Examples (TF-IDF + Logistic Regression) ---

--- Misclassification #1 ---
Original Tweet: @JetBlue should I expect delays at dca for 7am departure?  Going to Orlando on flight 723
Cleaned Tweet: should i expect delays at dca for 7am departure? going to orlando on flight 723
True Label: neutral
Predicted Label: negative

--- Misclassification #2 ---
Original Tweet: @JetBlue Thank you for credits. However; I submitted complaints about the property on vacation package. Hope you listen!
Cleaned Tweet: thank you for credits. however; i submitted complaints about the property on vacation package. hope you listen!
True Label: negative
Predicted Label: positive

--- Misclassification #3 ---
Original Tweet: @AmericanAir Not here yet, but I plan on it. If you could have them fly low and slow right in front of me, that would be great. ;-)
Cleaned Tweet: not here yet, but i plan on

## 5. Model B: Deep Learning (LSTM)

In [21]:
#This block of code was originally adapated from AI 102 Module 5 Guided Lab, but the Module 5 guided lab was a character-level model.
#Therefore, also used the Google Gemini integration in Google Colab with supplemented error analysis and correction also from Google Gemini in Google Colab.
# This took about 20 minutes to run.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Label Encoding
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# 2. Text Tokenization and Padding
max_words = 10000  # Vocabulary size
max_len = 100    # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(clean_x_train)

x_train_seq = tokenizer.texts_to_sequences(clean_x_train)
x_val_seq = tokenizer.texts_to_sequences(clean_x_val)
x_test_seq = tokenizer.texts_to_sequences(clean_x_test)

x_train_padded = pad_sequences(x_train_seq, maxlen=max_len)
x_val_padded = pad_sequences(x_val_seq, maxlen=max_len)
x_test_padded = pad_sequences(x_test_seq, maxlen=max_len)

# 3. PyTorch Dataset and DataLoader
BATCH_SIZE = 64

train_data = TensorDataset(torch.LongTensor(x_train_padded), torch.LongTensor(y_train_encoded))
val_data = TensorDataset(torch.LongTensor(x_val_padded), torch.LongTensor(y_val_encoded))
test_data = TensorDataset(torch.LongTensor(x_test_padded), torch.LongTensor(y_test_encoded))

train_loader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_data, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE)

# 4. LSTM Model Definition
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers, bidirectional, dropout_rate):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout_rate if num_layers > 1 else 0, # Dropout only if num_layers > 1
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), output_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, text):
        # text = [batch size, seq len]
        embedded = self.dropout(self.embedding(text))
        # embedded = [batch size, seq len, embedding dim]

        output, (hidden, cell) = self.lstm(embedded)
        # output = [batch size, seq len, hidden dim * num directions]
        # hidden = [num layers * num directions, batch size, hidden dim]

        # Use the hidden state from the last layer (and combine directions if bidirectional)
        if self.lstm.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        # hidden = [batch size, hidden dim * num directions]

        return self.fc(hidden)

# Model parameters
VOCAB_SIZE = max_words # From tokenizer
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = len(label_encoder.classes_) # Number of unique sentiment classes
NUM_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT_RATE = 0.5

model = SentimentLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS, BIDIRECTIONAL, DROPOUT_RATE).to(device)

# 5. Training Loop
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters())

def train(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            predictions = model(texts)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item()

            all_preds.extend(predictions.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(loader), all_preds, all_labels

N_EPOCHS = 3 # Train for 3 epochs

print("\nStarting LSTM Training...")
for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss, _, _ = evaluate(model, val_loader, criterion)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')

# 6. Evaluation on Test Set
print("\nEvaluating on Test Set...")
test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion)

print(f'Test Loss: {test_loss:.3f}')

# Convert numerical predictions and labels back to original sentiment for reporting
original_test_labels = label_encoder.inverse_transform(test_labels)
original_test_preds = label_encoder.inverse_transform(test_preds)

print("\nClassification Report for LSTM Model:")
print(classification_report(original_test_labels, original_test_preds, digits=4))

accuracy = accuracy_score(original_test_labels, original_test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(original_test_labels, original_test_preds, average='weighted', zero_division=0)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted): {recall:.4f}")
print(f"F1-Score (weighted): {f1:.4f}")


Using device: cpu

Starting LSTM Training...
Epoch: 01 | Train Loss: 0.805 | Val Loss: 0.673
Epoch: 02 | Train Loss: 0.677 | Val Loss: 0.611
Epoch: 03 | Train Loss: 0.615 | Val Loss: 0.596

Evaluating on Test Set...
Test Loss: 0.570

Classification Report for LSTM Model:
              precision    recall  f1-score   support

    negative     0.8505    0.8556    0.8530      1835
     neutral     0.6188    0.6048    0.6117       620
    positive     0.6660    0.6702    0.6681       473

    accuracy                         0.7725      2928
   macro avg     0.7118    0.7102    0.7109      2928
weighted avg     0.7716    0.7725    0.7721      2928


Accuracy: 0.7725
Precision (weighted): 0.7716
Recall (weighted): 0.7725
F1-Score (weighted): 0.7721


In [22]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder # Assuming label_encoder is still in kernel

# Assuming test_preds, test_labels, x_test, clean_x_test, and label_encoder
# are available from previous cells, specifically tUJoWXkdKs6y.

# Convert test_preds and test_labels (which are lists of numpy ints) to numpy arrays
# for easier comparison, if they aren't already.
# Make sure test_preds and test_labels are flat arrays of numerical labels.
# If they are still lists, convert them:
if isinstance(test_preds, list):
    test_preds_np = np.array(test_preds)
else:
    test_preds_np = test_preds

if isinstance(test_labels, list):
    test_labels_np = np.array(test_labels)
else:
    test_labels_np = test_labels

# Identify misclassified examples
misclassified_indices_lstm = np.where(test_labels_np != test_preds_np)[0]

print(f"Found {len(misclassified_indices_lstm)} misclassified examples in the test set for LSTM.\n")

print("--- Sample of Misclassified Examples (LSTM) ---")
num_samples_to_show = 5

if len(misclassified_indices_lstm) == 0:
    print("No misclassified examples found. The model achieved 100% accuracy on the test set!")
else:
    for i, idx in enumerate(misclassified_indices_lstm[:num_samples_to_show]):
        original_text = x_test.iloc[idx]
        cleaned_text = clean_x_test.iloc[idx]
        true_label_encoded = test_labels_np[idx]
        predicted_label_encoded = test_preds_np[idx]

        # Decode labels using the existing label_encoder
        true_label = label_encoder.inverse_transform([true_label_encoded])[0]
        predicted_label = label_encoder.inverse_transform([predicted_label_encoded])[0]

        print(f"\n--- Misclassification #{i+1} ---")
        print(f"Original Tweet: {original_text}")
        print(f"Cleaned Tweet: {cleaned_text}")
        print(f"True Label: {true_label}")
        print(f"Predicted Label: {predicted_label}")

print("\n--- Analysis of Misclassification Patterns (LSTM) ---")
print("Deep learning models like LSTM, while capable of understanding more context than TF-IDF, still face challenges:")
print("1.  **Subtlety and Nuance:** LSTMs are better at sequences, but subtle cues, irony, or sarcasm can still be missed if not strongly represented in training data. For example, 'Great, my flight is delayed *again*' might be classified as positive due to 'great' if the model doesn't fully grasp the negative implication of 'again' in this context.")
print("2.  **Long-Range Dependencies:** Although LSTMs are designed to handle long sequences, very long and complex sentences where the sentiment-determining word appears far from its subject can still pose a challenge, leading to loss of critical information.")
print("3.  **Ambiguous Language:** Tweets that are genuinely ambiguous, or where the sentiment shifts mid-sentence (e.g., 'The service was terrible, but the staff tried their best'), can confuse the model. It might focus on the initial negative part or the later positive part, leading to an incorrect overall sentiment.")
print("4.  **Imbalanced Classes:** If the model has seen significantly fewer examples of 'neutral' or 'positive' sentiments (as is common in this dataset), it might struggle to accurately classify these minority classes, sometimes defaulting to the majority 'negative' class.")
print("5.  **Out-of-Vocabulary (OOV) Words:** Despite embedding layers, entirely new or rare words not adequately captured during tokenization or in the embedding space can hinder understanding, especially if they carry strong sentiment.")
print("These misclassifications highlight the complexity of human language and the continuous need for better architectures, larger and more balanced datasets, and improved training techniques in deep learning for NLP.")

Found 666 misclassified examples in the test set for LSTM.

--- Sample of Misclassified Examples (LSTM) ---

--- Misclassification #1 ---
Original Tweet: @united This link in your tweet goes to someone's internal email -&gt; http://t.co/ZksX79itdN...... Probably one of your 3rd party IT contracts
Cleaned Tweet: this link in your tweet goes to someone's internal email -&gt; probably one of your 3rd party it contracts
True Label: neutral
Predicted Label: negative

--- Misclassification #2 ---
Original Tweet: @united too long for 140 characters
Cleaned Tweet: too long for 140 characters
True Label: negative
Predicted Label: positive

--- Misclassification #3 ---
Original Tweet: @USAirways Well I did miss it. But gate agents had rebooked boarding pass waiting when I landed. Time for lunch &amp; a beverage. Easy cheesy
Cleaned Tweet: well i did miss it. but gate agents had rebooked boarding pass waiting when i landed. time for lunch &amp; a beverage. easy cheesy
True Label: positive
Predict

EMBEDDINGS

The LSTM model used an nn.Embedding layer in PyTorch, which creates a lookup table for words in the vocabulary. These embeddings were randomly initialized and then learned from scratch during the training process of the LSTM model itself, rather than being pre-trained embeddings like Word2Vec or GloVe.

Found 666 misclassified examples in the test set for LSTM.

--- Sample of Misclassified Examples (LSTM) ---

--- Misclassification #1 ---
Original Tweet: @united This link in your tweet goes to someone's internal email -&gt; http://t.co/ZksX79itdN...... Probably one of your 3rd party IT contracts
Cleaned Tweet: this link in your tweet goes to someone's internal email -&gt; probably one of your 3rd party it contracts
True Label: neutral
Predicted Label: negative

--- Misclassification #2 ---
Original Tweet: @united too long for 140 characters
Cleaned Tweet: too long for 140 characters
True Label: negative
Predicted Label: positive

--- Misclassification #3 ---
Original Tweet: @USAirways Well I did miss it. But gate agents had rebooked boarding pass waiting when I landed. Time for lunch &amp; a beverage. Easy cheesy
Cleaned Tweet: well i did miss it. but gate agents had rebooked boarding pass waiting when i landed. time for lunch &amp; a beverage. easy cheesy
True Label: positive
Predicted Label: negative

--- Misclassification #4 ---
Original Tweet: @SouthwestAir My flight was 952, leaving las vegas at 5:40pm, arriving at CHI-MID at 11:00 pm.
Cleaned Tweet: my flight was 952, leaving las vegas at 5:40pm, arriving at chi-mid at 11:00 pm.
True Label: negative
Predicted Label: neutral

--- Misclassification #5 ---
Original Tweet: @JetBlue such a bummer.  But I understand it's a business deal. Thanks for answering me!  Much less sad now.
Cleaned Tweet: such a bummer. but i understand it's a business deal. thanks for answering me! much less sad now.
True Label: neutral
Predicted Label: positive

--- Analysis of Misclassification Patterns (LSTM) ---
Deep learning models like LSTM, while capable of understanding more context than TF-IDF, still face challenges:
1.  **Subtlety and Nuance:** LSTMs are better at sequences, but subtle cues, irony, or sarcasm can still be missed if not strongly represented in training data. For example, 'Great, my flight is delayed *again*' might be classified as positive due to 'great' if the model doesn't fully grasp the negative implication of 'again' in this context.
2.  **Long-Range Dependencies:** Although LSTMs are designed to handle long sequences, very long and complex sentences where the sentiment-determining word appears far from its subject can still pose a challenge, leading to loss of critical information.
3.  **Ambiguous Language:** Tweets that are genuinely ambiguous, or where the sentiment shifts mid-sentence (e.g., 'The service was terrible, but the staff tried their best'), can confuse the model. It might focus on the initial negative part or the later positive part, leading to an incorrect overall sentiment.
4.  **Imbalanced Classes:** If the model has seen significantly fewer examples of 'neutral' or 'positive' sentiments (as is common in this dataset), it might struggle to accurately classify these minority classes, sometimes defaulting to the majority 'negative' class.
5.  **Out-of-Vocabulary (OOV) Words:** Despite embedding layers, entirely new or rare words not adequately captured during tokenization or in the embedding space can hinder understanding, especially if they carry strong sentiment.
These misclassifications highlight the complexity of human language and the continuous need for better architectures, larger and more balanced datasets, and improved training techniques in deep learning for NLP.

1st Time:

Starting LSTM Training...
Epoch: 01 | Train Loss: 0.796 | Val Loss: 0.701
Epoch: 02 | Train Loss: 0.677 | Val Loss: 0.601
Epoch: 03 | Train Loss: 0.616 | Val Loss: 0.594

Evaluating on Test Set...
Test Loss: 0.569

Classification Report for LSTM Model:
              precision    recall  f1-score   support

    negative     0.8568    0.8316    0.8440      1835
     neutral     0.5425    0.7000    0.6113       620
    positive     0.7695    0.5645    0.6512       473

    accuracy                         0.7606      2928
   macro avg     0.7229    0.6987    0.7022      2928
weighted avg     0.7762    0.7606    0.7636      2928


Accuracy: 0.7606
Precision (weighted): 0.7762
Recall (weighted): 0.7606
F1-Score (weighted): 0.7636

2nd time:

Run Time: 26 min

Using device: cpu

Starting LSTM Training...
Epoch: 01 | Train Loss: 0.805 | Val Loss: 0.673
Epoch: 02 | Train Loss: 0.677 | Val Loss: 0.611
Epoch: 03 | Train Loss: 0.615 | Val Loss: 0.596

Evaluating on Test Set...
Test Loss: 0.570

Classification Report for LSTM Model:
              precision    recall  f1-score   support

    negative     0.8505    0.8556    0.8530      1835
     neutral     0.6188    0.6048    0.6117       620
    positive     0.6660    0.6702    0.6681       473

    accuracy                         0.7725      2928
   macro avg     0.7118    0.7102    0.7109      2928
weighted avg     0.7716    0.7725    0.7721      2928


Accuracy: 0.7725
Precision (weighted): 0.7716
Recall (weighted): 0.7725
F1-Score (weighted): 0.7721

In [ ]:
!pip install keras

## 6. Model C: LLM (BERT/DistilBERT fine-tuning)

In [ ]:
!pip install transformers accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
import pandas as pd # Import pandas

# Re-encode labels for Hugging Face Trainer compatibility, ensuring it matches the model output_dim
# The previous label_encoder is for PyTorch, so we need to ensure consistency or re-do if necessary
# Assuming y_train, y_val, y_test are still Series with original labels.

# Ensure labels are integers starting from 0 for the Hugging Face model
# First, combine all labels to fit the encoder, then transform subsets
all_labels = pd.concat([y_train, y_val, y_test])
label_encoder_hf = LabelEncoder()
label_encoder_hf.fit(all_labels)

train_labels_encoded = label_encoder_hf.transform(y_train)
val_labels_encoded = label_encoder_hf.transform(y_val)
test_labels_encoded = label_encoder_hf.transform(y_test)

num_classes = len(label_encoder_hf.classes_)
print(f"Number of classes: {num_classes}")

tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Correcting variable names and using encoded labels
train_ds = Dataset.from_dict({'text': clean_x_train.tolist(), 'label': train_labels_encoded.tolist()})
val_ds  = Dataset.from_dict({'text': clean_x_val.tolist(), 'label': val_labels_encoded.tolist()})
test_ds  = Dataset.from_dict({'text': clean_x_test.tolist(),  'label': test_labels_encoded.tolist()})

def tokenize(b):
  # Removed padding=True here, let the Trainer's DataCollatorWithPadding handle it dynamically
  return tok(b['text'], truncation=True, max_length=256)

ds_tok = DatasetDict({
    'train': train_ds.map(tokenize, batched=True),
    'validation': val_ds.map(tokenize, batched=True),
    'test': test_ds.map(tokenize, batched=True)
})

# Correcting num_labels to use the dynamically determined number of classes
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_classes)

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)
  p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
  acc = accuracy_score(labels, preds)
  return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

args = TrainingArguments(output_dir='./out', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=32, eval_strategy='epoch', learning_rate=2e-5, weight_decay=0.01)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'], # Use validation set for evaluation during training
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tok) # Explicitly provide data collator
)

print("\nStarting LLM Fine-tuning...")
trainer.train()

print("\nEvaluating LLM on Test Set...")
llm_eval_results = trainer.evaluate(eval_dataset=ds_tok['test'])
print(llm_eval_results)

# Optionally, extract and print individual metrics for comparison
llm_accuracy = llm_eval_results.get('eval_accuracy')
llm_precision = llm_eval_results.get('eval_precision')
llm_recall = llm_eval_results.get('eval_recall')
llm_f1 = llm_eval_results.get('eval_f1')

print(f"\nLLM Test Accuracy: {llm_accuracy:.4f}")
print(f"LLM Test Precision (weighted): {llm_precision:.4f}")
print(f"LLM Test Recall (weighted): {llm_recall:.4f}")
print(f"LLM Test F1-Score (weighted): {llm_f1:.4f}")

Number of classes: 3


Map:   0%|          | 0/12810 [00:00<?, ? examples/s]

Map:   0%|          | 0/1830 [00:00<?, ? examples/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Evaluating LLM on Test Set...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.327063,0.370994,2,0.865779,0.864933,0.865779,0.865296


{'eval_loss': 0.3709944486618042, 'eval_accuracy': 0.8657786885245902, 'eval_precision': 0.864933206610755, 'eval_recall': 0.8657786885245902, 'eval_f1': 0.8652961764894624}

LLM Test Accuracy: 0.8658
LLM Test Precision (weighted): 0.8649
LLM Test Recall (weighted): 0.8658
LLM Test F1-Score (weighted): 0.8653


References:

Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., Cistac, P., Rault, T., Louf, R., Funtowicz, M., Davison, J., Shleifer, S., von Platen, P., Ma, C., Jernite, Y., Plu, J., Xu, C., Le Scao, T., Gugger, S., … Rush, A. M. (2020). Transformers: State-of-the-art natural language processing. In Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations (pp. 38–45). Association for Computational Linguistics. https://doi.org/10.18653/v1/2020.emnlp-demos.6

Epoch	Training Loss	Validation Loss	Accuracy	Precision	Recall	F1
1	0.541662	0.454034	0.821311	0.827214	0.821311	0.823739
2	0.327063	0.451944	0.832240	0.831132	0.832240	0.831586

## 7. Results comparison
Create a table of metrics for all models (Accuracy/Precision/Recall/F1).

In [ ]:
import pandas as pd

# --- TF-IDF + Logistic Regression Metrics (from cell xACEQdZsETry output) ---
# From the output:
#              precision    recall  f1-score   support
#    negative     0.8166    0.9590    0.8821      1147
#     neutral     0.7182    0.5387    0.6156       388
#    positive     0.8333    0.5424    0.6571       295
#    accuracy                         0.8027      1830
#   macro avg     0.7894    0.6800    0.7183      1830
# weighted avg     0.7985    0.8027    0.7893      1830

tfidf_lr_accuracy = 0.8027
tfidf_lr_precision = 0.7985 # weighted avg
tfidf_lr_recall = 0.8027   # weighted avg (same as accuracy in this report)
tfidf_lr_f1 = 0.7893     # weighted avg

# --- LSTM Model Metrics (from cell tUJoWXkdKs6y output) ---
# From the output:
# Accuracy: 0.7606
# Precision (weighted): 0.7762
# Recall (weighted): 0.7606
# F1-Score (weighted): 0.7636

lstm_accuracy = 0.7606
lstm_precision = 0.7762
lstm_recall = 0.7606
lstm_f1 = 0.7636

# --- DistilBERT Model Metrics (from llm_eval_results in kernel state) ---
# The variables `llm_accuracy`, `llm_precision`, `llm_recall`, `llm_f1` are available in the kernel state
distilbert_accuracy = llm_accuracy
distilbert_precision = llm_precision
distilbert_recall = llm_recall
distilbert_f1 = llm_f1

# Create a DataFrame for comparison
results = pd.DataFrame({
    'Model': ['TF-IDF + Logistic Regression', 'LSTM', 'DistilBERT'],
    'Accuracy': [tfidf_lr_accuracy, lstm_accuracy, distilbert_accuracy],
    'Precision (weighted)': [tfidf_lr_precision, lstm_precision, distilbert_precision],
    'Recall (weighted)': [tfidf_lr_recall, lstm_recall, distilbert_recall],
    'F1-Score (weighted)': [tfidf_lr_f1, lstm_f1, distilbert_f1]
})

print("\n--- Model Comparison ---")
display(results.set_index('Model').round(4))


--- Model Comparison ---


,Accuracy,Precision (weighted),Recall (weighted),F1-Score (weighted)
Model,,,,
TF-IDF + Logistic Regression,0.8027,0.7985,0.8027,0.7893
LSTM,0.7606,0.7762,0.7606,0.7636
DistilBERT,0.8658,0.8649,0.8658,0.8653


Model
TF-IDF + Logistic Regression 0.8027 0.7893
LSTM 0.7606 0.7636
DistilBERT 0.8658 0.8653
From these results, it's clear that the DistilBERT model performed the best across all metrics (Accuracy, Precision, Recall, and F1-Score). It achieved an accuracy of approximately 86.58%.

The TF-IDF + Logistic Regression model came in second, with an accuracy of around 80.27%.

The LSTM model had the lowest performance among the three, with an accuracy of approximately 76.06%.

This trend is generally expected, as pre-trained large language models like DistilBERT often capture more nuanced semantic information, leading to better performance in text classification tasks compared to traditional machine learning (TF-IDF + Logistic Regression) or simpler deep learning models (LSTM) when fine-tuned on specific datasets.


## 8. Error analysis & reflection
Show misclassified examples and discuss patterns. Connect to challenges (context, emojis, domain shift, explainability).

In [ ]:
print("\nGetting predictions for error analysis...")
llm_predictions_output = trainer.predict(ds_tok['test'])
llm_test_preds_logits = llm_predictions_output.predictions
llm_test_preds_encoded = np.argmax(llm_test_preds_logits, axis=1)

# True labels are already available as test_labels_encoded
llm_test_true_encoded = test_labels_encoded

# Identify misclassified indices
misclassified_indices = np.where(llm_test_preds_encoded != llm_test_true_encoded)[0]

print(f"Found {len(misclassified_indices)} misclassified examples in the test set.")

# Map encoded labels back to original sentiment strings
original_llm_test_true_labels = label_encoder_hf.inverse_transform(llm_test_true_encoded)
original_llm_test_preds_labels = label_encoder_hf.inverse_transform(llm_test_preds_encoded)

print("\n--- Sample of Misclassified Examples (DistilBERT) ---")
num_samples_to_show = 10

if len(misclassified_indices) == 0:
    print("No misclassified examples found. The model achieved 100% accuracy on the test set!")
else:
    for i, idx in enumerate(misclassified_indices[:num_samples_to_show]):
        original_text = x_test.iloc[idx]
        cleaned_text = clean_x_test.iloc[idx]
        true_label = original_llm_test_true_labels[idx]
        predicted_label = original_llm_test_preds_labels[idx]

        print(f"\n--- Misclassification #{i+1} ---")
        print(f"Original Tweet: {original_text}")
        print(f"Cleaned Tweet: {cleaned_text}")
        print(f"True Label: {true_label}")
        print(f"Predicted Label: {predicted_label}")


Getting predictions for error analysis...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Found 393 misclassified examples in the test set.

--- Sample of Misclassified Examples (DistilBERT) ---

--- Misclassification #1 ---
Original Tweet: @JetBlue would you say a delay is more likely? Thanks so much.
Cleaned Tweet: would you say a delay is more likely? thanks so much.
True Label: positive
Predicted Label: neutral

--- Misclassification #2 ---
Original Tweet: @united This link in your tweet goes to someone's internal email -&gt; http://t.co/ZksX79itdN...... Probably one of your 3rd party IT contracts
Cleaned Tweet: this link in your tweet goes to someone's internal email -&gt; probably one of your 3rd party it contracts
True Label: neutral
Predicted Label: negative

--- Misclassification #3 ---
Original Tweet: @SouthwestAir BETSY is the BESTY! Gettin' stuck at #LAS might not be bad for most..but I want home! #homewardbound #betsy #besty #thankyou
Cleaned Tweet: betsy is the besty! gettin' stuck at #las might not be bad for most..but i want home! #homewardbound #betsy #best

### Error Analysis and Reflection

From the sample of misclassified examples, we can observe various reasons why the DistilBERT model, despite its strong performance, might make errors. Text classification, especially sentiment analysis, presents several inherent challenges:

1.  **Context and Nuance:**
    *   Tweets are often short, informal, and highly contextual. Sarcasm, irony, or subtle nuances in language can be difficult for models to grasp without extensive real-world understanding. For example, a tweet like "Great, another *delay* from this airline" might be predicted as neutral or even positive if the model over-indexes on 'great' and misses the sarcastic intent implied by 'delay'.

2.  **Emojis (and their removal):**
    *   While we decided to remove emojis due to their low frequency (3% of tweets), emojis can carry significant sentiment. If an emoji-less tweet is neutral but was originally accompanied by a strong negative emoji, its removal might strip away crucial sentiment cues, leading to misclassification.

3.  **Domain Shift (Implicit):**
    *   Although the dataset is specific to airline sentiment, the real-world language used by customers can be diverse. If the pre-training data for DistilBERT or our fine-tuning data didn't adequately cover certain expressions, jargon, or complaints unique to airline travel, the model might struggle. For instance, specific airline-related terms or complaint structures might not be perfectly mapped.

4.  **Imbalance in Classes:**
    *   As noted earlier, the dataset is imbalanced towards 'negative' sentiment. While the model generally performs well, it might still have a slight bias towards predicting the majority class, or conversely, struggle with the minority classes ('neutral' and 'positive') which have fewer examples to learn from.

5.  **Ambiguity and Subjectivity:**
    *   Human-labeled data itself can be subjective. What one person considers 'neutral' another might interpret as 'slightly negative' or 'slightly positive'. This inherent ambiguity in the ground truth can contribute to errors, as the model tries to learn from potentially inconsistent labels.

6.  **Explainability:**
    *   Large language models like DistilBERT are often considered "black boxes." When a misclassification occurs, it's challenging to pinpoint exactly *why* the model made that specific prediction. Was it a particular word? A sequence of words? The model's internal representation? Tools like LIME or SHAP can offer some insight, but full transparency remains an active research area.

**Further Steps for Improvement:**

*   **Data Augmentation:** Generate more examples for minority classes or paraphrased versions of existing examples to improve robustness.
*   **Advanced Preprocessing:** Experiment with keeping or encoding emojis, or more sophisticated techniques for handling informal language.
*   **Ensemble Methods:** Combine predictions from multiple models (e.g., TF-IDF, LSTM, DistilBERT) to leverage their diverse strengths.
*   **Hyperparameter Tuning:** More extensive tuning of model architectures and training parameters for all models, especially the deep learning ones.
*   **Error Categorization:** Manually categorize misclassification types (e.g., sarcasm, incorrect entity recognition, subtle negative tone) to identify systematic weaknesses.

## 9. Reproducibility notes (how to run)
List environment, hardware (CPU/GPU), and exact commands used.

In [17]:
# LSTM Model Losses (from tUJoWXkdKs6y output - last epoch values)
lstm_train_loss = 0.616
lstm_val_loss = 0.594
lstm_test_loss = 0.569

# DistilBERT Model Losses (from -_tHGGQ1vGKl text cell and Wl5erNc5ETrz output)
# Training and Validation Losses are from Epoch 2 in text cell -_tHGGQ1vGKl.
# The Test Loss ('eval_loss') for DistilBERT was printed as part of 'llm_eval_results' in cell Wl5erNc5ETrz,
# but that output was truncated and 'eval_loss' is not explicitly available as a named kernel variable.
# Therefore, it is marked as 'N/A' based on the provided context.
distilbert_train_loss = 0.327063
distilbert_val_loss = 0.451944
distilbert_test_loss = 'N/A'

# Create a DataFrame for comparison
loss_results = pd.DataFrame({
    'Model': ['LSTM', 'DistilBERT'],
    'Training Loss (last epoch)': [lstm_train_loss, distilbert_train_loss],
    'Validation Loss (last epoch)': [lstm_val_loss, distilbert_val_loss],
    'Test Loss': [lstm_test_loss, distilbert_test_loss]
})

print("--- Model Loss Comparison ---")
display(loss_results.set_index('Model'))

--- Model Loss Comparison ---


,Training Loss (last epoch),Validation Loss (last epoch),Test Loss
Model,,,
LSTM,0.616000,0.594000,0.569
DistilBERT,0.327063,0.451944,N/A


### Environment & Libraries

To reproduce the environment, install the following key libraries with their versions (or newer compatible versions).


In [ ]:
print('Python Version:')
!python --version

print('\nKey Library Versions:')
!pip show pandas numpy scikit-learn torch transformers tensorflow datasets accelerate


Python Version:
Python 3.13.15

Key Library Versions:
Name: pandas
Version: 2.2.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
All rights reserved.

Copyright (c) 2011-2023, Open source contributors.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

* Neither the name of the copyrig

### Hardware Information

The models were trained on a CPU or GPU, depending on availability. The DistilBERT model leveraged the `accelerate` library which can automatically detect and utilize available GPUs.


In [ ]:
print('CPU Information:')
!lscpu | grep 'Model name\|Architecture\|CPU(s):'

print('\nGPU Information (if available):')
!nvidia-smi


CPU Information:
Architecture:                            x86_64
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
NUMA node0 CPU(s):                       0,1

GPU Information (if available):
/bin/bash: line 1: nvidia-smi: command not found


### Exact Installation Commands

To ensure all necessary libraries are installed, the following commands were used:


In [ ]:
pip install pandas numpy scikit-learn matplotlib seaborn
pip install torch
pip install tensorflow keras
pip install transformers accelerate -U
pip uninstall -y datasets  # Uninstall datasets to clear any corrupted state
pip install --upgrade datasets # Reinstall/upgrade datasets
pip install nltk
pip install keras

In addition to these installations, the NLTK data (stopwords and wordnet) was downloaded programmatically within the notebook using `nltk.download('stopwords')` and `nltk.download('wordnet')`.

# References:

Abadi, M., Agarwal, A., Barham, P., Brevdo, E., Chen, Z., Citro, C., Corrado, G. S., Davis, A., Dean, J., Devin, M., Ghemawat, S., Goodfellow, I., Harp, A., Irving, G., Isard, M., Jia, Y., Jozefowicz, R., Kaiser, Ł., Kudlur, M., … Zheng, X. (2016). TensorFlow: Large-scale machine learning on heterogeneous systems. arXiv. https://arxiv.org/abs/1603.04467

Bird, S., Klein, E., & Loper, E. (2009). Natural language processing with Python. O’Reilly Media. https://www.nltk.org/book/

Chollet, F., & others. (2015). Keras. https://keras.io

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., … Oliphant, T. E. (2020). Array programming with NumPy. Nature, 585(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. Computing in Science & Engineering, 9(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

Kaggle. Twitter US Airline Sentiment. https:www.kaggle.com/crowdflower/twitter-airline-sentiment. Accessed on September 19, 2026.

Lhoest, Q., Delangue, C., Moi, A., Cistac, P., Rault, T., Louf, R., Funtowicz, M., Davison, J., von Platen, P., Patil, S., Chaumond, J., Drame, M., Plu, J., Xu, C., Le Scao, T., Gugger, S., Drame, M., Tunstall, L., Debut, L., … Wolf, T. (2021). Datasets: A community library for natural language processing. In Proceedings of the 2021 Conference on Empirical Methods in Natural Language Processing: System Demonstrations (pp. 175–184). Association for Computational Linguistics. https://doi.org/10.18653/v1/2021.emnlp-demo.21

McKinney, W. (2010). Data structures for statistical computing in Python. In S. van der Walt & J. Millman (Eds.), Proceedings of the 9th Python in Science Conference (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a

Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., Killeen, T., Lin, Z., Gimelshein, N., Antiga, L., Desmaison, A., Köpf, A., Yang, E., DeVito, Z., Raison, M., Tejani, A., Chilamkurthy, S., Steiner, B., Fang, L., … Chintala, S. (2019). PyTorch: An imperative style, high-performance deep learning library. In H. Wallach, H. Larochelle, A. Beygelzimer, F. d’Alché-Buc, E. Fox, & R. Garnett (Eds.), Advances in Neural Information Processing Systems, 32 (pp. 8024–8035). Curran Associates, Inc. https://proceedings.neurips.cc/paper/2019/hash/bdbca288fee7f92f2bfa9f7012727740-Abstract.html

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., VanderPlas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825–2830. http://jmlr.org/papers/v12/pedregosa11a.html

Python Software Foundation. (2024). string — Common string operations. In Python 3 documentation. https://docs.python.org/3/library/string.html

Waskom, M. L. (2021). seaborn: Statistical data visualization. Journal of Open Source Software, 6(60), 3021. https://doi.org/10.21105/joss.03021

Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., Cistac, P., Rault, T., Louf, R., Funtowicz, M., Davison, J., Shleifer, S., von Platen, P., Ma, C., Jernite, Y., Plu, J., Xu, C., Le Scao, T., Gugger, S., … Rush, A. M. (2020). Transformers: State-of-the-art natural language processing. In Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations (pp. 38–45). Association for Computational Linguistics. https://doi.org/10.18653/v1/2020.emnlp-demos.6